In [1]:
import pandas as pd
import re
import requests
import time
import os

In [2]:
# List of cards that are allowed to have multiple copies in a Commander deck
SINGLETON_EXCEPTIONS = {
    "Shadowborn Apostle", "Persistent Petitioners", "Rat Colony", 
    "Dragon's Approach", "Slime Against Humanity", "Relentless Rats"
}

### This if for fetching single cards for testing purposes
def get_scryfall_data(card_name):
    try:
        url = f"https://api.scryfall.com/cards/named?fuzzy={requests.utils.quote(card_name)}"
        response = requests.get(url, timeout=10)
        return response.json() if response.status_code == 200 else None
    except Exception:
        return None

def get_bulk_scryfall_data_with_fuzzy_fallback(card_names):
    """
    1. Tries to fetch cards in bulk (fastest).
    2. Identifies which names were 'not_found'.
    3. Runs a single fuzzy fetch for only the missing cards.
    """
    url_bulk = "https://api.scryfall.com/cards/collection"
    headers = {"User-Agent": "MTG-Project/1.0", "Content-Type": "application/json"}
    
    # --- PASS 1: BULK FETCH ---
    payload = {"identifiers": [{"name": n} for n in card_names]}
    found_cards = []
    missing_names = []

    try:
        res = requests.post(url_bulk, json=payload, headers=headers, timeout=20)
        if res.status_code == 200:
            data = res.json()
            found_cards = data.get('data', [])
            # Scryfall returns a 'not_found' list of identifiers that failed exact match
            not_found_objects = data.get('not_found', [])
            missing_names = [obj.get('name') for obj in not_found_objects if 'name' in obj]
    except Exception as e:
        print(f"❌ Bulk API Error: {e}")
        missing_names = card_names # Fallback to all if bulk fails

    # --- PASS 2: FUZZY FALLBACK (For DFCs and Flavor Names) ---
    if missing_names:
        print(f"🔍 Fixing {len(missing_names)} unknowns...")
        for name in missing_names:
            try:
                fuzzy_url = f"https://api.scryfall.com/cards/named?fuzzy={requests.utils.quote(name)}"
                f_res = requests.get(fuzzy_url, headers=headers, timeout=10)
                if f_res.status_code == 200:
                    card_data = f_res.json()
                    
                    # CRITICAL FIX: 
                    # Map the data using the 'name' from your FILE, 
                    # not the 'name' from Scryfall.
                    scryfall_results[name] = card_data 
                    
                    print(f"   ✅ Recovered: {name} (as {card_data.get('name')})")
                else:
                    print(f"   ❌ Still Unknown: {name}")
            except:
                continue
    return found_cards

def parse_complex_mana(cost_str):
    if not cost_str:
        return {'w':0,'u':0,'b':0,'r':0,'g':0,'c':0,'x':0,'phi_opt':0,'hybrids':[]}
    symbols = re.findall(r'\{([^}]+)\}', cost_str)
    data = {'w':0,'u':0,'b':0,'r':0,'g':0,'c':0,'x':0,'phi_opt':0,'hybrids':[]}
    for s in [sym.upper() for sym in symbols]:
        if s in 'WUBRG': data[s.lower()] += 1
        elif s == 'C': data['c'] += 1
        elif s == 'X': data['x'] += 1
        elif s.isdigit(): data['c'] += int(s)
        elif '/P' in s:
            data[s.replace('/P', '').lower()] += 1
            data['phi_opt'] += 2
        elif '/' in s:
            data['hybrids'].append(s)
            first = s.split('/')[0].lower()
            if first in data: data[first] += 1
            elif first.isdigit(): data['c'] += int(first)
    return data

def process_commander_deck_to_100(input_file):
    deck_id = os.path.basename(input_file).replace('.txt', '')
    print(f"--- 🔍 PROCESSING DECK: {deck_id} ---")
    
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]

      # 1. Parse Name/Qty and Clean the names
    card_entries = []
    for i, line in enumerate(lines):
        match = re.match(r'^(\d+)\s+(.*)$', line)
        if match:
            qty, name = int(match.group(1)), match.group(2).strip()
            
            # --- FIX B: Sanitize Split Cards ---
            # Converts 'Wear/Tear' to 'Wear // Tear' for API compatibility
            if "/" in name and " // " not in name:
                name = name.replace("/", " // ")
            
            is_comm = (i == len(lines) - 1)
            card_entries.append({'name': name, 'qty': qty, 'is_commander': is_comm})

    # 2. Bulk Fetch card data from Scryfall
    all_names = [e['name'] for e in card_entries]
    # Scryfall allows 75 per batch, we do 100 in two small batches
    batch_1 = all_names[:75]
    batch_2 = all_names[75:]
    
    raw_data_list = get_bulk_scryfall_data_with_fuzzy_fallback(batch_1) + get_bulk_scryfall_data_with_fuzzy_fallback(batch_2)
    # Map names to data for easy lookup
    scryfall_map = {card['name']: card for card in raw_data_list if 'name' in card}

    # 3. Build the Final Deck List with all your original columns
    deck_list = []
    for entry in card_entries:
        name = entry['name']
        raw = scryfall_map.get(name, {})
        
        # Handle multi-faced cards (Transform/MDFC)
        face = raw.get('card_faces', [raw])[0]
        mana_to_parse = raw.get('mana_cost', face.get('mana_cost', ''))
        
        # Parse mana using your original function
        p = parse_complex_mana(raw.get('mana_cost', face.get('mana_cost', '')))
        
        card_data = {
            'deck_id': deck_id,
            'name': raw.get('name', name), # Fallback to input name if fetch failed
            'type_line': raw.get('type_line', 'Unknown'),
            'cmc': int(raw.get('cmc', 0)),
            'mana_cost': raw.get('mana_cost', face.get('mana_cost', '')),
            'w': p['w'], 'u': p['u'], 'b': p['b'], 'r': p['r'], 'g': p['g'], 'c': p['c'],
            'x_count': p['x'], 'phi_life_opt': p['phi_opt'], 'hybrid_opts': "|".join(p['hybrids']),
            'effects_line': raw.get('oracle_text', face.get('oracle_text', '')).replace('\n', ' '),
            'is_commander': entry['is_commander']
        }
        
        # Expand by quantity to reach 100 rows
        for _ in range(entry['qty']):
            deck_list.append(card_data)

    df = pd.DataFrame(deck_list)
    print(f"📊 Final Count for {deck_id}: {len(df)} cards.")
    return df

In [3]:
files = ["moxfield_export_1.txt", "moxfield_export_2.txt", "moxfield_export_3.txt", "moxfield_export_4.txt", "moxfield_export_5.txt", "moxfield_export_6.txt", "moxfield_export_7.txt"]
all_decks = []
for f in files:
    if os.path.exists(f):
        all_decks.append(process_commander_deck_to_100(f))

# Combine and save
master_df = pd.concat(all_decks)

--- 🔍 PROCESSING DECK: moxfield_export_1 ---
🔍 Fixing 1 unknowns...
📊 Final Count for moxfield_export_1: 100 cards.
--- 🔍 PROCESSING DECK: moxfield_export_2 ---
🔍 Fixing 1 unknowns...
📊 Final Count for moxfield_export_2: 100 cards.
--- 🔍 PROCESSING DECK: moxfield_export_3 ---
🔍 Fixing 3 unknowns...
📊 Final Count for moxfield_export_3: 100 cards.
--- 🔍 PROCESSING DECK: moxfield_export_4 ---
🔍 Fixing 1 unknowns...
📊 Final Count for moxfield_export_4: 100 cards.
--- 🔍 PROCESSING DECK: moxfield_export_5 ---
🔍 Fixing 2 unknowns...
🔍 Fixing 1 unknowns...
📊 Final Count for moxfield_export_5: 100 cards.
--- 🔍 PROCESSING DECK: moxfield_export_6 ---
🔍 Fixing 1 unknowns...
📊 Final Count for moxfield_export_6: 100 cards.
--- 🔍 PROCESSING DECK: moxfield_export_7 ---
🔍 Fixing 2 unknowns...
📊 Final Count for moxfield_export_7: 100 cards.


In [4]:
master_df.to_csv('mtg_enriched_data_master.csv', index=False)
print(f"✨ Done! Total rows: {len(master_df)}")

✨ Done! Total rows: 700


In [5]:
def verify_master_file(file_path='mtg_enriched_data_master.csv'):
    print("\n" + "="*50)
    print("🔍 FINAL MASTER FILE AUDIT")
    print("="*50)
    
    if not os.path.exists(file_path):
        print(f"🚨 ERROR: {file_path} not found.")
        return

    df_check = pd.read_csv(file_path)
    
    # 1. Count Unknowns
    # A card is 'Unknown' if the fetch failed or the type/effects are empty
    unknowns = df_check[
        (df_check['type_line'] == 'Unknown') | 
        (df_check['effects_line'].isna()) | 
        (df_check['effects_line'] == '')
    ]
    
    total_cards = len(df_check)
    unknown_count = len(unknowns)
    
    # 2. Check Deck Integrity (Expecting 100 per deck)
    deck_counts = df_check['deck_id'].value_counts()
    
    print(f"📊 Total Rows in Master: {total_cards}")
    print(f"❓ Total Unknowns Found: {unknown_count}")
    
    if unknown_count > 0:
        print("\n⚠️  LIST OF UNKNOWN CARDS (Action Required):")
        # Grouping by name so we don't list the same unknown 30 times
        unique_unknowns = unknowns.groupby(['deck_id', 'name']).size().reset_index(name='count')
        for _, row in unique_unknowns.iterrows():
            print(f"   - [{row['deck_id']}] {row['name']} ({row['count']} copies)")
    else:
        print("\n✅ Clean Sweep: No Unknown cards detected.")

    print("\n📦 DECK COMPOSITION CHECK:")
    for d_id, count in deck_counts.items():
        status = "✅ PERFECT" if count == 100 else "❌ ERROR"
        print(f"   - {d_id}: {count}/100 cards ({status})")
        
    print("="*50)

In [6]:
verify_master_file()


🔍 FINAL MASTER FILE AUDIT
📊 Total Rows in Master: 700
❓ Total Unknowns Found: 26

⚠️  LIST OF UNKNOWN CARDS (Action Required):
   - [moxfield_export_1] Agadeem's Awakening (1 copies)
   - [moxfield_export_1] Reckless Stormseeker (1 copies)
   - [moxfield_export_1] Sisters of the Undead (1 copies)
   - [moxfield_export_1] The Restoration of Eiganjo (1 copies)
   - [moxfield_export_2] Eirdu, Carrier of Dawn (1 copies)
   - [moxfield_export_2] Matzalantli, the Great Door (1 copies)
   - [moxfield_export_2] Sheoldred (1 copies)
   - [moxfield_export_2] Wear // Tear (1 copies)
   - [moxfield_export_3] Cordyceps Excision (1 copies)
   - [moxfield_export_3] Eirdu, Carrier of Dawn (1 copies)
   - [moxfield_export_3] Porom's Silence Magic (1 copies)
   - [moxfield_export_3] Shantotto's Coercion (1 copies)
   - [moxfield_export_4] Wear // Tear (1 copies)
   - [moxfield_export_4] Witch Enchanter (1 copies)
   - [moxfield_export_5] Aang's Shelter (1 copies)
   - [moxfield_export_5] Emet-Selch, Un

In [7]:
# 1. Define your name corrections (File Name -> Scryfall Search Name)
corrections = {
    "Sisters of the Undead": "Olivia, Crimson Bride",
    "Mayor Tong of Chin Village": "Drannith Magistrate",
    "Aang's Shelter": "Teferi's Protection",
    "Emet-Selch, Unsundered": "Talion, the Kindly Lord",
    "Revival/Revenge": "Revival // Revenge"
}

def final_cleanup_pass(df, correction_map):
    print("🧹 Starting final cleanup pass...")
    
    # Identify rows that are still 'Unknown'
    unknown_indices = df[df['type_line'] == 'Unknown'].index
    
    for idx in unknown_indices:
        original_name = df.loc[idx, 'name']
        if isinstance(original_name, pd.Series):
            original_name = original_name.iloc[0]
        
        # Determine the best name to search for
        search_name = correction_map.get(original_name, original_name)
        
        print(f"🔄 Re-fetching: {original_name} (as '{search_name}')...")
        
        # Use your single fuzzy fetcher
        raw = get_scryfall_data(search_name) # Ensure this uses the 'fuzzy' logic we discussed
        
        if raw:
            face = raw.get('card_faces', [raw])[0]
            # Re-parse mana for the new data
            p = parse_complex_mana(raw.get('mana_cost', face.get('mana_cost', '')))
            
            # Update the row with the real data
            df.at[idx, 'type_line'] = raw.get('type_line')
            df.at[idx, 'cmc'] = int(raw.get('cmc', 0))
            df.at[idx, 'effects_line'] = raw.get('oracle_text', face.get('oracle_text', '')).replace('\n', ' ')
            df.at[idx, 'w'] = p['w']
            df.at[idx, 'u'] = p['u']
            df.at[idx, 'b'] = p['b']
            df.at[idx, 'r'] = p['r']
            df.at[idx, 'g'] = p['g']
            # ... update any other columns you need ...
            
            print(f"   ✅ Successfully updated {original_name}!")
        else:
            print(f"   ❌ Could not resolve {original_name}. Check spelling or custom card status.")
            
    return df

In [8]:
# Execute the cleanup
master_df = final_cleanup_pass(master_df, corrections)

# Save the final, finalized version
master_df.to_csv('mtg_enriched_data_master_FINAL.csv', index=False)

🧹 Starting final cleanup pass...
🔄 Re-fetching: Agadeem's Awakening (as 'Agadeem's Awakening')...
   ✅ Successfully updated Agadeem's Awakening!
🔄 Re-fetching: Reckless Stormseeker (as 'Reckless Stormseeker')...
   ✅ Successfully updated Reckless Stormseeker!
🔄 Re-fetching: Sisters of the Undead (as 'Olivia, Crimson Bride')...
   ✅ Successfully updated Sisters of the Undead!
🔄 Re-fetching: The Restoration of Eiganjo (as 'The Restoration of Eiganjo')...
   ✅ Successfully updated The Restoration of Eiganjo!
🔄 Re-fetching: Deafening Silence (as 'Deafening Silence')...
   ✅ Successfully updated Deafening Silence!
🔄 Re-fetching: Hushbringer (as 'Hushbringer')...
   ✅ Successfully updated Hushbringer!
🔄 Re-fetching: Rakshasa Debaser (as 'Rakshasa Debaser')...
   ✅ Successfully updated Rakshasa Debaser!
🔄 Re-fetching: Vault of Champions (as 'Vault of Champions')...
   ✅ Successfully updated Vault of Champions!
🔄 Re-fetching: Breena, the Demagogue (as 'Breena, the Demagogue')...
   ✅ Successfu

In [9]:
verify_master_file('mtg_enriched_data_master_FINAL.csv')


🔍 FINAL MASTER FILE AUDIT
📊 Total Rows in Master: 700
❓ Total Unknowns Found: 7

⚠️  LIST OF UNKNOWN CARDS (Action Required):
   - [moxfield_export_5] Osgiliath, Fallen Capital (1 copies)
   - [moxfield_export_5] Sink into Stupor (1 copies)
   - [moxfield_export_5] Unstable Harmonics (1 copies)
   - [moxfield_export_6] Cordyceps Excision (1 copies)
   - [moxfield_export_6] Sundering Eruption (1 copies)
   - [moxfield_export_7] Mayor Tong of Chin Village (1 copies)
   - [moxfield_export_7] Revival // Revenge (1 copies)

📦 DECK COMPOSITION CHECK:
   - moxfield_export_1: 100/100 cards (✅ PERFECT)
   - moxfield_export_2: 100/100 cards (✅ PERFECT)
   - moxfield_export_3: 100/100 cards (✅ PERFECT)
   - moxfield_export_4: 100/100 cards (✅ PERFECT)
   - moxfield_export_5: 100/100 cards (✅ PERFECT)
   - moxfield_export_6: 100/100 cards (✅ PERFECT)
   - moxfield_export_7: 100/100 cards (✅ PERFECT)


In [10]:
# Configuration
MAX_ATTEMPTS = 3
attempt = 1

while attempt <= MAX_ATTEMPTS:
    # Count current unknowns
    unknown_count = len(master_df[master_df['type_line'] == 'Unknown'])
    
    if unknown_count == 0:
        print(f"✅ All cards resolved after {attempt-1} extra passes.")
        break
        
    print(f"\n🔄 Pass {attempt}/{MAX_ATTEMPTS}: Attempting to resolve {unknown_count} unknowns...")
    
    # Run the cleanup pass
    master_df = final_cleanup_pass(master_df, corrections)
    
    # Optional: Increase sleep time between passes to let the Scryfall rate-limit reset
    time.sleep(2) 
    attempt += 1


🔄 Pass 1/3: Attempting to resolve 7 unknowns...
🧹 Starting final cleanup pass...
🔄 Re-fetching: Night's Whisper (as 'Night's Whisper')...
   ✅ Successfully updated Night's Whisper!
🔄 Re-fetching: Serra Ascendant (as 'Serra Ascendant')...
   ✅ Successfully updated Serra Ascendant!
🔄 Re-fetching: Talisman of Hierarchy (as 'Talisman of Hierarchy')...
   ✅ Successfully updated Talisman of Hierarchy!
🔄 Re-fetching: Captain Lannery Storm (as 'Captain Lannery Storm')...
   ✅ Successfully updated Captain Lannery Storm!
🔄 Re-fetching: Shadowblood Ridge (as 'Shadowblood Ridge')...
   ✅ Successfully updated Shadowblood Ridge!
🔄 Re-fetching: Fire Covenant (as 'Fire Covenant')...
   ✅ Successfully updated Fire Covenant!
🔄 Re-fetching: Plains (as 'Plains')...
   ✅ Successfully updated Plains!
✅ All cards resolved after 1 extra passes.


In [11]:
master_df.to_csv('mtg_enriched_data_master_FINAL.csv', index=False)

In [12]:
verify_master_file('mtg_enriched_data_master_FINAL.csv')


🔍 FINAL MASTER FILE AUDIT
📊 Total Rows in Master: 700
❓ Total Unknowns Found: 0

✅ Clean Sweep: No Unknown cards detected.

📦 DECK COMPOSITION CHECK:
   - moxfield_export_1: 100/100 cards (✅ PERFECT)
   - moxfield_export_2: 100/100 cards (✅ PERFECT)
   - moxfield_export_3: 100/100 cards (✅ PERFECT)
   - moxfield_export_4: 100/100 cards (✅ PERFECT)
   - moxfield_export_5: 100/100 cards (✅ PERFECT)
   - moxfield_export_6: 100/100 cards (✅ PERFECT)
   - moxfield_export_7: 100/100 cards (✅ PERFECT)
